# Web Scraping for Business Analysts – My Finance BA Version
Forked from JP Morgan Chase python-training repo  
Customized by S33mi (March 2026)  
Goal: Learn ethical web scraping with pandas + BeautifulSoup to extract financial tables, company data, and market info – super useful for BA reports, competitor analysis, or economic research.

# Accessing data from a website
Not all websites make it easy to grab data. Luckily, `pandas` can help.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

#result = requests.get('https://en.wikipedia.org/wiki/List_of_sovereign_states') # List_of_sovereign_states webpage has been modified and very complex structure
#pd.read_html(result.content)[0].head(20)

# 2026 update: Always use a browser-like User-Agent (many sites now block default requests)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36'
}

URL = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)" #Using simple wiki url
result = requests.get(URL, headers=headers)
pd.read_html(result.content)[0].head(20)

,Country or territory,Population (1 July 2022),Population (1 July 2023),Change (%),UN continental region[1],UN statistical subregion[1]
0,World,8021407192,8091734930,+0.88%,–,–
1,India,1425423212,1438069596,+0.89%,Asia,Southern Asia
2,China[a],1425179569,1422584933,−0.18%,Asia,Eastern Asia
3,United States,341534046,343477335,+0.57%,Americas,Northern America
4,Indonesia,278830529,281190067,+0.85%,Asia,South-eastern Asia
5,Pakistan,243700667,247504495,+1.56%,Asia,Southern Asia
6,Nigeria,223150896,227882945,+2.12%,Africa,Western Africa
7,Brazil,210306415,211140729,+0.40%,Americas,South America
8,Bangladesh,169384897,171466990,+1.23%,Asia,Southern Asia
9,Russia,145579899,145440500,−0.10%,Europe,Eastern Europe


For more complex parsing, we can utilize the `BeautifulSoup` library. Let's try to extract the same table, but use the new library.

In [2]:
soup = BeautifulSoup(result.content, 'lxml') # Parse the HTML as a string
str(soup)[:500]

'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled wp25easte'

Find the tables.

In [3]:
tables = soup.find_all('table')

Using the `read_html` function of `pandas`, read the first table into a dataframe.

In [4]:
pd.read_html(str(tables[0]))[0].head(20)

/tmp/ipykernel_152/1136478351.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_html(str(tables[0]))[0].head(20)


,Country or territory,Population (1 July 2022),Population (1 July 2023),Change (%),UN continental region[1],UN statistical subregion[1]
0,World,8021407192,8091734930,+0.88%,–,–
1,India,1425423212,1438069596,+0.89%,Asia,Southern Asia
2,China[a],1425179569,1422584933,−0.18%,Asia,Eastern Asia
3,United States,341534046,343477335,+0.57%,Americas,Northern America
4,Indonesia,278830529,281190067,+0.85%,Asia,South-eastern Asia
5,Pakistan,243700667,247504495,+1.56%,Asia,Southern Asia
6,Nigeria,223150896,227882945,+2.12%,Africa,Western Africa
7,Brazil,210306415,211140729,+0.40%,Americas,South America
8,Bangladesh,169384897,171466990,+1.23%,Asia,Southern Asia
9,Russia,145579899,145440500,−0.10%,Europe,Eastern Europe


As we can see, the data we get back isn't always perfect, which is what's so nice about APIs instead of parsing HTML. Nevertheless, we would benefit a lot if we simplified this into a function.

In [5]:
"""
def dfFromURL(url, tableNumber=1):
    soup = BeautifulSoup(requests.get(url).content, 'lxml') # Parse the HTML as a string
    tables = soup.find_all('table')
    # check table number is within number of tables on the page
    assert len(tables) >= tableNumber
    return pd.read_html(str(tables[tableNumber-1]))[0]
"""

"\ndef dfFromURL(url, tableNumber=1):\n    soup = BeautifulSoup(requests.get(url).content, 'lxml') # Parse the HTML as a string\n    tables = soup.find_all('table')\n    # check table number is within number of tables on the page\n    assert len(tables) >= tableNumber\n    return pd.read_html(str(tables[tableNumber-1]))[0]\n"

In [6]:
def dfFromURL(url, table_number=1):
    """
    Extract the Nth HTML table from a webpage into a pandas DataFrame.
    2026 Colab version: better headers, timeout, error handling.
    """
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.content, 'lxml')
        tables = soup.find_all('table')

        if len(tables) < table_number:
            raise ValueError(f"Only {len(tables)} tables found (asked for #{table_number})")

        # pd.read_html can take string or bytes
        df = pd.read_html(str(tables[table_number-1]))[0]
        print(f"Success! Table #{table_number} shape: {df.shape}")
        return df

    except requests.exceptions.RequestException as e:
        print(f"Network/request error: {e}")
    except ValueError as e:
        print(f"Parsing error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

    return None

Now we can make a pretty simple call to get an HTML table as a dataframe. Let's try it.

Yahoo Finance historical page (2026 note): This often fails now because the table is loaded via JavaScript.
The code below tries anyway — if it returns empty → use `yfinance` library instead (recommended).

In [33]:
from pandas.plotting import table
#prices = dfFromURL('https://finance.yahoo.com/quote/JPM/history')
#prices.head()
#prices

yahoo_url = 'https://finance.yahoo.com/quote/JPM/history/'

prices = dfFromURL(yahoo_url, table_number = 1)

if prices is not None:
    display(prices.head(10))
else:
    print("→ Tip: Install and use yfinance instead (much more reliable in 2026)")

Network/request error: 404 Client Error: Not Found for url: https://finance.yahoo.com/quote/JPM/history/
→ Tip: Install and use yfinance instead (much more reliable in 2026)


If you got some messy data hear with divs and some disclaimers on the bottom...let's clean it up with a simple `dropna`.


Otherwise move to the next cell
Install and use yfinance instead (much more reliable in 2026)

In [34]:
#Run this cell is you got some data
#prices = prices.dropna()
#prices.head()

In [35]:
import yfinance as yf

ticker = "JPM"
print(f"Downloading {ticker} data via yfinance...")

data = yf.download(ticker, period="1y", progress=False)   # or start="2025-01-01"
#data = data.dropna()
#data.head()
data.tail(15)   # most recent rows

Price,Close,High,Low,Open,Volume
Ticker,JPM,JPM,JPM,JPM,JPM
Date,,,,,
2026-02-13,302.549988,304.290009,296.519989,298.519989,9114500
2026-02-17,307.130005,308.239990,302.500000,302.760010,8896800
2026-02-18,308.779999,312.279999,307.220001,308.459991,7209600
2026-02-19,308.049988,309.179993,305.119995,307.170013,6737700
2026-02-20,310.790009,311.000000,305.679993,308.399994,7792700
2026-02-23,297.670013,311.000000,295.100006,308.799988,12955400
2026-02-24,297.299988,299.750000,291.380005,296.820007,13554100
2026-02-25,303.299988,303.660004,297.010010,298.640015,8095500


Cool! Let's try to get the second table from a website. Let's see what the Cavs record was for the last few seasons:
    

In [17]:
df1 = dfFromURL('https://www.espn.com/nba/team/stats/_/name/cle', 0)
df1

Success! Table #0 shape: (22, 14)


/tmp/ipykernel_152/1120126267.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables[table_number-1]))[0]


,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,2PM,2PA,2P%,SC-EFF,SH-EFF
0,9.9,20.6,48.3,3.5,9.4,36.9,5.2,6.1,85.2,6.5,11.2,57.8,1.388,0.57
1,5.9,12.6,46.9,2.9,6.3,45.6,4.4,5.3,83.3,3.0,6.2,48.2,1.522,0.58
2,6.5,14.5,45.1,2.3,6.3,36.0,2.6,3.0,86.1,4.3,8.2,52.1,1.239,0.53
3,6.8,13.1,51.7,1.2,3.7,31.5,2.9,4.6,63.2,5.6,9.4,59.6,1.342,0.56
4,6.0,9.4,63.6,0.0,0.2,10.0,3.3,4.5,72.5,6.0,9.2,64.8,1.620,0.64
5,4.7,11.2,42.3,1.7,5.5,30.8,2.8,3.2,86.9,3.0,5.7,53.5,1.245,0.50
6,5.1,10.2,50.3,2.1,4.6,45.8,1.3,1.7,75.3,3.0,5.6,53.8,1.333,0.60
7,4.4,9.1,47.8,3.4,7.3,46.1,1.1,1.2,85.4,1.0,1.8,54.2,1.438,0.66
8,3.6,9.0,39.8,0.9,2.8,32.4,2.7,3.1,86.5,2.7,6.2,43.2,1.194,0.45
9,2.5,5.0,49.1,1.0,3.3,30.6,0.4,0.5,80.0,1.5,1.7,84.2,1.255,0.59


In [18]:
df2 = dfFromURL('https://www.espn.com/nba/team/stats/_/name/cle', 1)
df2

Success! Table #1 shape: (22, 1)


/tmp/ipykernel_152/1120126267.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables[table_number-1]))[0]


,Name
0,Donovan Mitchell G
1,James Harden G *
2,Darius Garland G *
3,Evan Mobley C
4,Jarrett Allen C
5,De'Andre Hunter F *
6,Jaylon Tyson G
7,Sam Merrill G
8,Dennis Schroder G *
9,Keon Ellis G *


In [19]:
pd.concat((df2, df1), axis=1)

#End of Orignal Notebook avaiable at https://github.com/jpmorganchase/python-training/blob/main/notebooks/5_website.ipynb

,Name,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,2PM,2PA,2P%,SC-EFF,SH-EFF
0,Donovan Mitchell G,9.9,20.6,48.3,3.5,9.4,36.9,5.2,6.1,85.2,6.5,11.2,57.8,1.388,0.57
1,James Harden G *,5.9,12.6,46.9,2.9,6.3,45.6,4.4,5.3,83.3,3.0,6.2,48.2,1.522,0.58
2,Darius Garland G *,6.5,14.5,45.1,2.3,6.3,36.0,2.6,3.0,86.1,4.3,8.2,52.1,1.239,0.53
3,Evan Mobley C,6.8,13.1,51.7,1.2,3.7,31.5,2.9,4.6,63.2,5.6,9.4,59.6,1.342,0.56
4,Jarrett Allen C,6.0,9.4,63.6,0.0,0.2,10.0,3.3,4.5,72.5,6.0,9.2,64.8,1.620,0.64
5,De'Andre Hunter F *,4.7,11.2,42.3,1.7,5.5,30.8,2.8,3.2,86.9,3.0,5.7,53.5,1.245,0.50
6,Jaylon Tyson G,5.1,10.2,50.3,2.1,4.6,45.8,1.3,1.7,75.3,3.0,5.6,53.8,1.333,0.60
7,Sam Merrill G,4.4,9.1,47.8,3.4,7.3,46.1,1.1,1.2,85.4,1.0,1.8,54.2,1.438,0.66
8,Dennis Schroder G *,3.6,9.0,39.8,0.9,2.8,32.4,2.7,3.1,86.5,2.7,6.2,43.2,1.194,0.45
9,Keon Ellis G *,2.5,5.0,49.1,1.0,3.3,30.6,0.4,0.5,80.0,1.5,1.7,84.2,1.255,0.59


### Easy Table Scraping with pandas.read_html – Finance Example
Let's scrape Wikipedia's "List of largest companies" table (public data, great for BA market research).

In [20]:
#New Cell Added By S33mi
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

import requests
import pandas as pd
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [21]:
# Grokipedia-specific headers (helps avoid blocks + shows educational intent)
headers = {
    'User-Agent': 'Mozilla/5.0 (Educational Project by S33mi) - Personal Learning Notebook'
}

# Test connection
test_url = "https://grokipedia.com/"
response = requests.get(test_url, headers=headers)
print("Grokipedia status:", response.status_code)
if response.status_code == 200:
    print("Ready to scrape Grokipedia!")

Grokipedia status: 200
Ready to scrape Grokipedia!


### Easy Table Scraping with pandas.read_html – Grokipedia Example
Let's pull the "List of largest companies" or similar table from Grokipedia.

In [36]:
grok_url = "https://grokipedia.com/page/List_of_largest_companies_by_revenue"  # Or search for exact page title
# If the exact URL is different, use: https://grokipedia.com/search?q=List+of+largest+companies

#tables = pd.read_html(grok_url, headers=headers)
tables = pd.read_html(grok_url, storage_options=headers) #pandas >= 2.1
if tables:
    companies_grok = tables[0]  # Usually the main table is index 0
    display(companies_grok.head(10))
    print(f"Rows scraped from Grokipedia: {len(companies_grok)}")
    companies_grok.to_csv('largest_companies_grokipedia.csv', index=False)
else:
    print("No tables found – try a different page or check URL.")

,Rank,Company,Country,Industry,Revenue (millions USD)
0,1,Walmart,United States,Retail,680985
1,2,Amazon,United States,Retail/Technology,637959
2,3,State Grid Corporation of China,China,Utilities,548414
3,4,Saudi Aramco,Saudi Arabia,Energy,480194
4,5,China National Petroleum,China,Energy,412645
5,6,Sinopec Group,China,Energy,407490
6,7,UnitedHealth Group,United States,Healthcare,400278
7,8,Apple,United States,Technology,391035
8,9,CVS Health,United States,Healthcare,372809
9,10,Berkshire Hathaway,United States,Conglomerate,371433


Rows scraped from Grokipedia: 10


In [37]:
response = requests.get(grok_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Extract page title (Grokipedia often has unique AI-generated intros)
title = soup.find('h1').get_text() if soup.find('h1') else "No title found"
print("Page Title:", title)

# Find all links in the article body
links = soup.find_all('a')[:15]  # First 15 for demo
for link in links:
    text = link.get_text().strip()
    href = link.get('href')
    if text and href and not href.startswith('#'):
        print(f"{text} → {href}")

# Extract any tables manually
grok_tables = soup.find_all('table')
if grok_tables:
    df_bs = pd.read_html(str(grok_tables[0]))[0]
    display(df_bs.head())

Page Title: List of largest companies by revenue
Search âK → /search
Sign in → https://accounts.x.ai/check-login?redirect=grokipedia-com&return_to=%2Fpage%2FList_of_largest_companies_by_revenue
Sign in → https://accounts.x.ai/check-login?redirect=grokipedia-com&return_to=%2Fpage%2FList_of_largest_companies_by_revenue


,Rank,Company,Country,Industry,Revenue (millions USD)
0,1,Walmart,United States,Retail,680985
1,2,Amazon,United States,Retail/Technology,637959
2,3,State Grid Corporation of China,China,Utilities,548414
3,4,Saudi Aramco,Saudi Arabia,Energy,480194
4,5,China National Petroleum,China,Energy,412645


## USA-Focused: Scrape Grokipedia for "Economy of USA" or "List of banks in USA"

In [56]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

us_url = "https://grokipedia.com/page/List_of_largest_banks_in_the_United_States"

# Try this first — pandas can handle most pages directly
tables = pd.read_html(us_url, storage_options=headers)   # pandas ≥ 1.5/2.0+ supports storage_options

# Or fallback (very reliable):
# response = requests.get(us_url, headers=headers)
# tables = pd.read_html(response.text)

print(f"Found {len(tables)} tables")

if tables:
    # The main list of largest banks by assets is usually the first wikitable → index 0
    df_assets = tables[0]
    df_to_export = tables[0] #will use this for expoting data to csv
    print("Top banks by total assets:")
    display(df_assets.head(10))   # or just df_assets

    # There are also tables for deposits, market cap, etc. → check index 1, 2...

Found 5 tables
Top banks by total assets:


,Rank,Bank Name (Affiliated Holding Company),"Headquarters (City, State)",Total Assets ($ millions)
0,1,"JPMorgan Chase Bank, N.A. (JPMorgan Chase & Co.)","Columbus, OH",3788551
1,2,"Bank of America, N.A. (Bank of America Corp.)","Charlotte, NC",2665555
2,3,"Citibank, N.A. (Citigroup Inc.)","Sioux Falls, SD",1833933
3,4,"Wells Fargo Bank, N.A. (Wells Fargo & Co.)","Sioux Falls, SD",1746394
4,5,"U.S. Bank, N.A. (U.S. Bancorp)","Cincinnati, OH",671438
5,6,"Capital One, N.A. (Capital One Financial Corp.)","McLean, VA",648909
6,7,Goldman Sachs Bank USA (The Goldman Sachs Grou...,"New York, NY",625410
7,8,"PNC Bank, N.A. (PNC Financial Services Group, ...","Wilmington, DE",554573
8,9,Truist Bank (Truist Financial Corp.),"Charlotte, NC",535585
9,10,The Bank of New York Mellon (Bank of New York ...,"New York, NY",398293


#Extracting HTML pages using `BeautifulSoup`

In [47]:
us_url = "https://grokipedia.com/page/List_of_largest_banks_in_the_United_States"  # Or search for it
us_response = requests.get(us_url, headers=headers)
us_soup = BeautifulSoup(us_response.text, 'html.parser')

# Quick table extraction if available
us_tables = pd.read_html(str(us_soup.find('table', {'class': 'wikitable'}))) if us_soup.find('table', {'class': 'wikitable'}) else [] #class = wikitable/ infobox
if us_tables:
    us_economy = us_tables[0]
    display(us_economy)
else:
    print("No infobox/wiki-table found using soup method– extract html text instead.")

# Extract key facts (e.g., GDP, growth rate)
facts = us_soup.find_all('p')[:5]
for p in facts:
    print(p.get_text().strip())

No infobox/wiki-table found using soup method– extract html text instead.
Suggest a topic for Grokipedia to cover. Our AI will research and create an article if suitable.
Tips for a good suggestion



The tables on https://grokipedia.com/page/List_of_largest_banks_in_the_United_States are not real HTML table elements — they are **Markdown tables**.

Grokipedia (like many modern wiki-style or rendered pages) stores and renders content using Markdown syntax:

| Rank | Bank Name ... | Headquarters ... | Total Assets ... |
|------|---------------|------------------|------------------|
| 1    | JPMorgan ...  | Columbus, OH     | 3,788,551        |
...

In [49]:
wikitables = soup.find_all('table', class_='wikitable')
print(f"Found {len(wikitables)} wikitables")

for i, tbl in enumerate(wikitables):
    df = pd.read_html(str(tbl))[0]
    print(f"\nTable {i}:")
    display(df.head())

Found 0 wikitables


BeautifulSoup finds zero table tags → everything returns `None` or empty list → your code thinks no table exists.

Don't use BeautifulSoup for table extraction here — just stick with `pd.read_html(response.text)` or `pd.read_html(url, storage_options=headers)`.
It's simpler, more reliable on this site, and already working perfectly for you.

So using classicala pandas `pd.read_html()` works most of the time.

#Extracting Markdown Tables using `BeautifulSoup` and `pandas`

If you insist on BeautifulSoup (e.g. to select a specific table by position, caption, or surrounding text):
Search for Markdown table markers instead, or
Let pandas do the heavy lifting after soup, like this:



In [48]:
url = "https://grokipedia.com/page/List_of_largest_banks_in_the_United_States"
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

# Option A: Let pandas parse the whole page (easiest & usually best)
all_tables = pd.read_html(response.text)   # or str(soup) — same thing
print(f"Found {len(all_tables)} tables")

# Main one is almost always tables[0]
if all_tables:
    df_main = all_tables[0]
    print("Main table (by total assets):")
    display(df_main.head(10))

# Option B: Try to isolate one Markdown table using surrounding text (more fragile)
# Example: find text near the heading, then extract the block
heading = soup.find(string=lambda t: "by consolidated total assets" in t.lower() if t else False)
if heading:
    # Walk forward to find the next pre/code/markdown block — but it's tricky
    # Often easier to just use pd.read_html on substrings, but not worth it here
    pass

Found 5 tables
Main table (by total assets):


,Rank,Bank Name (Affiliated Holding Company),"Headquarters (City, State)",Total Assets ($ millions)
0,1,"JPMorgan Chase Bank, N.A. (JPMorgan Chase & Co.)","Columbus, OH",3788551
1,2,"Bank of America, N.A. (Bank of America Corp.)","Charlotte, NC",2665555
2,3,"Citibank, N.A. (Citigroup Inc.)","Sioux Falls, SD",1833933
3,4,"Wells Fargo Bank, N.A. (Wells Fargo & Co.)","Sioux Falls, SD",1746394
4,5,"U.S. Bank, N.A. (U.S. Bancorp)","Cincinnati, OH",671438
5,6,"Capital One, N.A. (Capital One Financial Corp.)","McLean, VA",648909
6,7,Goldman Sachs Bank USA (The Goldman Sachs Grou...,"New York, NY",625410
7,8,"PNC Bank, N.A. (PNC Financial Services Group, ...","Wilmington, DE",554573
8,9,Truist Bank (Truist Financial Corp.),"Charlotte, NC",535585
9,10,The Bank of New York Mellon (Bank of New York ...,"New York, NY",398293


### Export Scraped Data to CSV – Ready for Analysis or Sharing
After scraping tables from Grokipedia (or any source), save the DataFrame to CSV.  
This creates a local file you can:
- Open in Excel/Google Sheets
- Use in future pandas analysis
- Share in reports or your portfolio

The file will be saved in your current Colab working directory (or download it via Files panel).

In [57]:
#──────────────────────────────────────────────────────────────
# Export the last scraped table to CSV (change variable name if needed)
# ──────────────────────────────────────────────────────────────

# Choose which DataFrame to export (update this line to match your last table)
#df_to_export.head(1)

# Create a clean, descriptive filename
table_name = "us_banks_grokipedia_data"   # Customize this part
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
filename = f"{table_name}_{timestamp}.csv"

# Export
df_to_export.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"Successfully exported {len(df_to_export)} rows to:")
print(f"→ {filename}")
print("\nYou can now:")
print("• Download from the Files panel (left sidebar in Colab)")
print("• Open in Excel / Google Sheets")
print("• Import back later: pd.read_csv('" + filename + "')")

Successfully exported 20 rows to:
→ us_banks_grokipedia_data_20260309_084447.csv

You can now:
• Download from the Files panel (left sidebar in Colab)
• Open in Excel / Google Sheets
• Import back later: pd.read_csv('us_banks_grokipedia_data_20260309_084447.csv')


In [58]:
# Bonus: Quick preview of what was saved
print("\nFirst 5 rows of exported data:")
display(df_to_export.head())

# Bonus: Auto-download in Colab (optional – uncomment if desired)
# from google.colab import files
# files.download(filename)


First 5 rows of exported data:


,Rank,Bank Name (Affiliated Holding Company),"Headquarters (City, State)",Total Assets ($ millions)
0,1,"JPMorgan Chase Bank, N.A. (JPMorgan Chase & Co.)","Columbus, OH",3788551
1,2,"Bank of America, N.A. (Bank of America Corp.)","Charlotte, NC",2665555
2,3,"Citibank, N.A. (Citigroup Inc.)","Sioux Falls, SD",1833933
3,4,"Wells Fargo Bank, N.A. (Wells Fargo & Co.)","Sioux Falls, SD",1746394
4,5,"U.S. Bank, N.A. (U.S. Bancorp)","Cincinnati, OH",671438


## Compare Grokipedia vs Wikipedia
- Run the same scrape on en.wikipedia.org equivalent and compare row count, columns, or content tone.
- Grokipedia often has more "truth-seeking" framing per xAI's philosophy.

## Key Takeaways & Exercises
- Mastered: Scraping AI-generated sites, headers/User-Agent, CSV export.
- Exercises:
  1. Scrape Grokipedia for "US Stock Exchange" and extract company list.
  2. Compare GDP figures from Grokipedia vs Wikipedia for "Economy of USA".
  3. Add error handling for 403/429 responses.
  4. Bonus: Use requests.Session() for multiple page scrapes.
Commit to your fork – this version stands out with 2026 relevance!